# 마스크 진단: v7 마스크가 Cholis Table III와 일치하는가?

**목적**: v7의 PSC + disk mask가 Cholis Table III의 'Masked fraction' 컬럼과 얼마나 일치하는지 확인.

**Cholis Table III 기준**:
- 0.275–0.357 GeV: **71.8%** masked
- 0.357–0.464 GeV: 62.9%
- 0.464–0.603 GeV: 52.2%
- 0.603–0.784 GeV: 38.5%
- 0.784–1.02 GeV: 29.2%
- 1.02–1.32 GeV: 23.4%
- 1.32–1.72 GeV: 19.0%
- 1.72–2.24 GeV: 16.3%
- 2.24–2.91 GeV: 13.0%
- 2.91–3.78 GeV: 12.9%
- 3.78–4.91 GeV: 11.6%
- 4.91–10.8 GeV: 11.5%
- 10.8–23.7 GeV: 10.3%
- 23.7–51.9 GeV: 10.3%

**가설**: 우리 마스크가 Cholis보다 훨씬 작음 (MASK_SCALE=0.7).
특히 저에너지에서 71.8%가 아니라 50%대 이하일 것으로 예상.
이게 사실이면 spectral shape 왜곡의 주요 원인.

In [ ]:
# %% ══════════════════════════════════════════════════════════
# 마스크 진단 1: 현재 마스크 vs Cholis Table III
# ══════════════════════════════════════════════════════════════

import numpy as np
import os
from astropy.io import fits

mask_file = os.path.join(WORK_DIR, 'masks_14bin.fits')

with fits.open(mask_file) as h:
    masks_3d = h[0].data

print(f'Mask shape: {masks_3d.shape}')

# Cholis paper의 masked fraction (Table III)
cholis_fractions = np.array([
    71.8, 62.9, 52.2, 38.5, 29.2, 23.4, 19.0, 16.3,
    13.0, 12.9, 11.6, 11.5, 10.3, 10.3,
])

S = OFFSET
E = S + NX_FIT

print('\n=== Mask 비교: 우리 v7 vs Cholis Table III (40x40 fitting region) ===')
print(f'{"ie":>3} {"E[GeV]":>9} {"우리[%]":>10} {"Cholis[%]":>11} {"차이[%p]":>10} {"비율":>8}')
print('-' * 65)

ours = []
for ie in range(N_ENERGY_BINS):
    m_crop = masks_3d[ie, S:E, S:E]
    frac_ours = 100.0 * (1 - m_crop.mean())
    ours.append(frac_ours)
    diff = frac_ours - cholis_fractions[ie]
    rat = frac_ours / cholis_fractions[ie]
    marker = '✅' if abs(diff) < 5 else ('🔴' if diff < -10 else '⚠️')
    print(f'{ie:3d} {ENERGY_CENTERS_GEV[ie]:9.3f} {frac_ours:10.1f} '
          f'{cholis_fractions[ie]:11.1f} {diff:+10.1f} {rat:8.3f} {marker}')

print()
print('해석:')
print('  ✅ 차이 < 5%p: Cholis와 일치 (좋음)')
print('  ⚠️ 차이 5~10%p: 약간 다름')
print('  🔴 차이 > 10%p (under-masking): 마스크가 너무 작음 — contamination 우려')
print()

# 가장 큰 deficit 빈
ours_arr = np.array(ours)
deficit = ours_arr - cholis_fractions
worst_idx = np.argmin(deficit)
print(f'\n가장 under-masking된 빈:')
print(f'  ie={worst_idx}, E={ENERGY_CENTERS_GEV[worst_idx]:.3f} GeV')
print(f'  우리: {ours_arr[worst_idx]:.1f}%, Cholis: {cholis_fractions[worst_idx]:.1f}%')
print(f'  차이: {deficit[worst_idx]:+.1f}%p')

In [ ]:
# %% ══════════════════════════════════════════════════════════
# 마스크 진단 2: 마스크 비율 vs ratio 상관 — under-masking이 0.155 deficit의 원인인가?
# ══════════════════════════════════════════════════════════════

# v7의 GCE ratio (사용자 출력에서)
v7_ratio = np.array([
    0.155, 0.486, 0.643, 0.790, 0.765, 0.775, 0.746,
    0.796, 0.780, 0.784, 0.832, 0.828, 0.932, 1.260
])

print('=== mask deficit vs GCE ratio 상관 ===')
print(f'{"ie":>3} {"E[GeV]":>9} {"mask_def[%p]":>14} {"GCE_ratio":>11}')
print('-' * 50)
for ie in range(N_ENERGY_BINS):
    print(f'{ie:3d} {ENERGY_CENTERS_GEV[ie]:9.3f} '
          f'{deficit[ie]:+14.1f} {v7_ratio[ie]:11.3f}')

# Pearson 상관
from scipy.stats import pearsonr
r, p = pearsonr(deficit, v7_ratio)
print(f'\nPearson r (mask_def, ratio) = {r:.3f}, p-value={p:.3e}')
print()
if r < -0.5:
    print('  → 음의 상관: mask가 작은 빈에서 ratio도 작음')
    print('  → 즉 under-masking이 GCE deficit과 직접 연결됨')
elif r > 0.5:
    print('  → 양의 상관: mask가 작은 빈에서 ratio가 큼 (의외)')
else:
    print('  → 약한 상관: under-masking이 spectral shape 왜곡의 주원인은 아닐 수 있음')

In [ ]:
# %% ══════════════════════════════════════════════════════════
# 마스크 진단 3: MASK_SCALE 변경 시 예상 영향 simulation
# ══════════════════════════════════════════════════════════════
# 만약 우리가 MASK_SCALE을 0.7 → 0.9로 바꾸면 어떻게 될까?
# 예상: 마스크 면적이 (0.9/0.7)² = 1.65배 증가
# 즉 마스크 비율이 ~1.65배 증가

scale_factor = (0.9 / 0.7) ** 2
print(f'MASK_SCALE 0.7 → 0.9 변경 시 예상 면적 변화: ×{scale_factor:.3f}')
print()

# 디스크 마스크 (|b|<2°)는 항상 ~10% 차지
disk_only_frac = 10.0  # 대략

print(f'{"ie":>3} {"E[GeV]":>9} {"v7 (0.7)":>10} {"예상 (0.9)":>11} {"Cholis":>9}')
print('-' * 55)
for ie in range(N_ENERGY_BINS):
    psc_v7 = max(ours_arr[ie] - disk_only_frac, 0)  # PSC contribution만
    psc_predicted = psc_v7 * scale_factor
    total_predicted = min(disk_only_frac + psc_predicted, 100)
    print(f'{ie:3d} {ENERGY_CENTERS_GEV[ie]:9.3f} {ours_arr[ie]:10.1f} '
          f'{total_predicted:11.1f} {cholis_fractions[ie]:9.1f}')

print()
print('이 시뮬레이션은 단순한 면적 scaling — 실제로는 source 분포에 따라 다를 수 있음')
print('하지만 대략적인 추정으로 MASK_SCALE 변경의 효과를 가늠할 수 있음')

In [ ]:
# %% ══════════════════════════════════════════════════════════
# 진단 4: 마스킹된 광자 비율 — 데이터 손실 확인
# ══════════════════════════════════════════════════════════════

with fits.open(ccube_file) as h:
    counts = h[0].data

print('=== 마스크 적용 후 photon counts 변화 ===')
print(f'{"ie":>3} {"E[GeV]":>9} {"raw[counts]":>14} {"masked[counts]":>16} {"손실[%]":>10}')
print('-' * 65)
for ie in range(N_ENERGY_BINS):
    raw = counts[ie, S:E, S:E].sum()
    masked_data = counts[ie, S:E, S:E] * masks_3d[ie, S:E, S:E]
    masked = masked_data.sum()
    loss = 100 * (1 - masked/raw) if raw > 0 else 0
    print(f'{ie:3d} {ENERGY_CENTERS_GEV[ie]:9.3f} {raw:14.0f} {masked:16.0f} {loss:10.1f}')

print()
print('비교: Cholis Table III masked fraction과 비슷해야 함')
print('만약 Cholis 71.8%이고 우리 50%면: 우리가 21.8%p 더 많은 데이터를 fit에 포함')
print('  → 그 추가 데이터는 PSC contamination이거나 sub-threshold source')
print('  → fit이 GCE template으로 그것들을 설명하려고 함')
print('  → GCE의 spectral shape 왜곡')